# 01 · Vendas da Vulto: raio-x

Fontes: `banco_vendas.csv` e `banco_itens_venda.csv` (gerados por `padronizar.py` em `staging_vulto/`).

Objetivo: **entender** as vendas da Vulto, sem decidir nem corrigir nada. Para cada fonte:

1. O que é cada linha?
2. O que significa cada coluna? (com exemplos)
3. Que problemas aparecem? (formatos, vazios, duplicados, valores estranhos)
4. Como ela se liga às outras fontes?

> Antes de commitar: *Kernel → Restart & Clear Output* (as tabelas têm CPF, e-mail e telefone).

In [ ]:
from pathlib import Path

import pandas as pd

# raiz do repositório, de onde quer que o notebook tenha sido aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "migracao_vulto").is_dir())
STAGING = RAIZ / "staging_vulto"

# mostra todas as colunas nas tabelas
pd.set_option("display.max_columns", None)

## 1. Carregar

`dtype=str` lê tudo como texto, do jeito que está no arquivo. Sem isso, o pandas tenta adivinhar o tipo de cada coluna e pode, por exemplo, apagar o zero à esquerda de um CPF.

In [ ]:
vendas = pd.read_csv(STAGING / "banco_vendas.csv", dtype=str)
itens = pd.read_csv(STAGING / "banco_itens_venda.csv", dtype=str)
lojas = pd.read_csv(STAGING / "banco_lojas.csv", dtype=str)

## 2. O que é cada linha?

In [ ]:
print("vendas:", vendas.shape, "| itens:", itens.shape)  # (linhas, colunas)
vendas.head()

### Vendas: cada linha é um pedido?

Se for, o `id` nunca se repete. E o `pedido_externo` (número do pedido no site ou no app) também não deveria se repetir.

In [ ]:
# se cada linha é um pedido, nenhum id se repete
print("ids repetidos:", vendas["id"].duplicated().sum())

# pedido_externo é o número do pedido no site/app: ele também deveria ser único
pedidos = vendas["pedido_externo"].dropna()
print("pedidos do site/app:", len(pedidos), "| repetidos:", pedidos.duplicated().sum())

In [ ]:
# exemplo de um pedido do app que aparece duas vezes
vendas[vendas["pedido_externo"] == "APP-7001324"]

### Itens: cada linha é uma peça dentro de um pedido?

`venda_id` diz a qual venda o item pertence.

In [ ]:
itens.head()

In [ ]:
# quantos pedidos têm 1 item, 2 itens, 3 itens...
itens.groupby("venda_id").size().value_counts().sort_index()

In [ ]:
# todo item aponta para uma venda que existe? toda venda tem pelo menos um item?
print("itens sem venda:", (~itens["venda_id"].isin(vendas["id"])).sum())
print("vendas sem item:", (~vendas["id"].isin(itens["venda_id"])).sum())

## 3. Coluna por coluna

### Grupo 1: onde a venda aconteceu (`canal`, `loja_id`, `vendedor`)

In [ ]:
# as lojas da Vulto (loja_id aponta para esta tabela)
lojas

In [ ]:
# canal: quais valores aparecem e quantas vezes
vendas["canal"].value_counts(dropna=False)

In [ ]:
# cruzamento: em cada loja, quais grafias de canal aparecem
pd.crosstab(vendas["canal"], vendas["loja_id"])

In [ ]:
# vendedor: quantas vendas sem vendedor em cada loja, e os nomes mais comuns
print("sem vendedor por loja:", vendas.groupby("loja_id")["vendedor"].apply(lambda s: s.isna().sum()).to_dict())
print("nomes diferentes:", vendas["vendedor"].nunique())
vendas["vendedor"].value_counts().head(15)

### Grupo 2: quem comprou (`cliente_ref`, `cliente_tipo`, `cliente_valor`)

In [ ]:
# cliente_tipo foi classificado pelo padronizar.py a partir do cliente_ref
display(pd.crosstab(vendas["cliente_tipo"].fillna("(vazio)"), vendas["loja_id"]))
# o que está escrito quando o cliente não foi identificado
vendas.loc[vendas["cliente_tipo"].isna(), "cliente_ref"].value_counts(dropna=False).head()

### Grupo 3: quanto (`total`, `desconto`, `frete`)

In [ ]:
total = pd.to_numeric(vendas["total"])
print("negativos:", (total < 0).sum(), "| acima de R$ 5.000 por loja:", vendas[total > 5000].groupby("loja_id").size().to_dict())
print("frete > 0 por loja:", vendas[pd.to_numeric(vendas["frete"]) > 0].groupby("loja_id").size().to_dict())
# no app (loja 6) a mediana passa de 22 mil: centavos gravados como se fossem reais
total[vendas["loja_id"] == "6"].describe()

### Grupo 4: situação e quando (`status`, `criado_em`, `obs`)

In [ ]:
print("status:", vendas["status"].value_counts(dropna=False).to_dict())
print("período (UTC):", vendas["criado_em"].min(), "a", vendas["criado_em"].max())
vendas["obs"].value_counts().head(20)

### Grupo 5: identificação (`pedido_externo`, `compra_id`)

In [ ]:
print("pedido_externo preenchido por loja:", vendas.groupby("loja_id")["pedido_externo"].apply(lambda s: s.notna().sum()).to_dict())
print("prefixos:", vendas["pedido_externo"].dropna().str[:3].value_counts().to_dict())
# compra_id liga a venda à tabela compras (pontos do Vulto+)
print("compra_id preenchido por loja:", vendas.groupby("loja_id")["compra_id"].apply(lambda s: s.notna().sum()).to_dict())

### Grupo 6: itens (`sku`, `descricao`, `qtd`, `preco_unit`)

In [ ]:
formato_sku = itens["sku"].map(lambda s: "longo (VLT-...)" if s.startswith("VLT-") else "com espaço" if " " in s
                               else "sem código" if not any(c.isdigit() for c in s) else "curto")
print("formato do SKU:", formato_sku.value_counts().to_dict())
print("qtd:", itens["qtd"].value_counts().to_dict())
# as quantidades grandes são de um pedido só
itens[pd.to_numeric(itens["qtd"]) > 10].merge(vendas[["id", "loja_id", "total", "obs"]], left_on="venda_id", right_on="id")

## 4. Que problemas aparecem?

A maioria apareceu nas seções anteriores (a lista completa está nas anotações). Falta um: vendas em que o `total` não é igual a itens − desconto + frete.

In [ ]:
# total esperado = soma dos itens - desconto + frete
valor_itens = pd.to_numeric(itens["qtd"]) * pd.to_numeric(itens["preco_unit"])
esperado = (vendas["id"].map(valor_itens.groupby(itens["venda_id"]).sum())
            - pd.to_numeric(vendas["desconto"]).fillna(0) + pd.to_numeric(vendas["frete"]).fillna(0))
diferenca = (pd.to_numeric(vendas["total"]) - esperado).round(2)
nao_batem = vendas[diferenca.abs() > 0.05]
print("não batem:", len(nao_batem), "| por loja:", nao_batem.groupby("loja_id").size().to_dict())
# as diferenças mais comuns: valores de frete, em reais (14.9) e em centavos (1490)
diferenca[diferenca.abs() > 0.05].value_counts().head(8)

### O app grava em centavos o tempo todo?

Comparando o preço de cada item com o preço do catálogo: perto de 1× está em reais; perto de 100×, em centavos.

In [ ]:
import re

# chave para comparar o mesmo SKU escrito de jeitos diferentes (VLT-CAM-004-BRA-M = CAM004BM); só para medir
def chave(sku):
    s = str(sku).upper().strip()
    longo = re.fullmatch(r"VLT[- ]([A-Z]{3})[- ](\d{3})[- ]([A-Z]+)[- ]?(.*)", s)
    if longo:
        return f"{longo[1]}{longo[2]}{longo[3][0]}{longo[4].replace(' ', '').replace('-', '')}"
    curto = re.fullmatch(r"([A-Z]{3})(\d{3})([A-Z])(.+)", s)
    if curto:
        return f"{curto[1]}{curto[2]}{curto[3]}{curto[4]}"
    return None

catalogo = pd.read_csv(STAGING / "catalogo_produtos.csv", dtype=str)
preco_catalogo = catalogo.assign(chave=catalogo["sku"].map(chave), p=pd.to_numeric(catalogo["preco"])).groupby("chave")["p"].max()
razao = pd.to_numeric(itens["preco_unit"]) / itens["sku"].map(chave).map(preco_catalogo)
escala = pd.cut(razao, [0, 5, 50, 1000], labels=["reais", "ambíguo", "centavos"]).astype(str).replace("nan", "sem preço no catálogo")
pd.crosstab(itens["venda_id"].map(vendas.set_index("id")["loja_id"]).rename("loja_id"), escala)

In [ ]:
# no app (loja 6), mês a mês: reais ou centavos?
app = vendas[vendas["loja_id"] == "6"]
mes = pd.to_datetime(app["criado_em"]).dt.strftime("%Y-%m")
pd.crosstab(mes, pd.to_numeric(app["total"]).abs() > 3000).rename(columns={False: "reais", True: "centavos"})

### As vendas com status vazio são pagas?

In [ ]:
# lojas físicas, mês a mês, por status
fisicas = vendas[vendas["loja_id"].isin(["1", "2", "3", "4"])]
status = fisicas["status"].fillna("(vazio)").str.strip().str.lower()
pd.crosstab(pd.to_datetime(fisicas["criado_em"]).dt.strftime("%Y-%m"), status).head(4)

## 5. Como se liga às outras fontes?

In [ ]:
catalogo = pd.read_csv(STAGING / "catalogo_produtos.csv", dtype=str)
estoque = pd.read_csv(STAGING / "banco_estoque.csv", dtype=str)
compras = pd.read_csv(STAGING / "banco_compras.csv", dtype=str)

print("loja_id existe em lojas:", f"{vendas['loja_id'].isin(lojas['id']).mean():.0%}")
print("compra_id existe em compras:", f"{vendas['compra_id'].dropna().str.removesuffix('.0').isin(compras['id']).mean():.0%}")
print("sku dos itens existe no estoque:", f"{itens['sku'].isin(estoque['sku']).mean():.0%}")
print("sku dos itens existe no catálogo (escrito igual):", f"{itens['sku'].isin(catalogo['sku']).mean():.0%}",
      "| códigos:", itens.loc[itens["sku"].isin(catalogo["sku"]), "sku"].nunique(), "de", itens["sku"].nunique())

In [ ]:
# cliente das vendas → planilha de clientes, por tipo de identificação
clientes = pd.read_csv(STAGING / "clientes_clientes.csv", dtype=str)
cpfs, emails, telefones = set(clientes["cpf"].dropna()), set(clientes["E-mail"].dropna().str.lower()), set(clientes["Tel"].dropna())

def acha_cliente(tipo, valor):
    if pd.isna(tipo):
        return "sem cliente"
    if tipo == "email":
        return "achou" if valor.lower() in emails else "não achou"
    if tipo in ("cpf", "telefone"):
        return "achou" if valor in (cpfs if tipo == "cpf" else telefones) else "não achou"
    if tipo == "numero":  # só dígitos: pode ser CPF ou celular
        return "achou como CPF" if valor in cpfs else "achou como telefone" if any(t.endswith(valor) for t in telefones) else "não achou"
    return "outro"

pd.crosstab(vendas["cliente_tipo"].fillna("(vazio)"),
            [acha_cliente(t, v) for t, v in zip(vendas["cliente_tipo"], vendas["cliente_valor"])])

In [ ]:
# atendimentos → vendas: o cliente informa o número do site/app ou um número solto (id interno)
atendimentos = pd.read_csv(STAGING / "atendimentos.csv", dtype=str)
codigos = atendimentos["pedido_codigo"].dropna()
print("atendimentos com pedido:", len(codigos),
      "| acham pelo número do site/app:", codigos.isin(vendas["pedido_externo"]).sum(),
      "| acham pelo id interno:", codigos.isin(vendas["id"]).sum(),
      "| não acham:", (~codigos.isin(set(vendas["pedido_externo"].dropna()) | set(vendas["id"]))).sum())

## Anotações para o dicionário de dados

**O que é cada linha**
- `vendas`: um pedido (o `id` nunca se repete). Exceção: os pedidos do app repetidos (abaixo).
- `itens`: uma peça (ou várias da mesma, pela `qtd`) dentro de um pedido. Liga na venda por `venda_id`; todo item tem venda e toda venda tem item. Pedidos têm de 1 a 5 itens.

**Colunas de `vendas`**

| Coluna | O que é | Exemplos | Problemas vistos |
|---|---|---|---|
| `loja_id` | unidade da venda (tabela `banco_lojas`) | 1–4 lojas físicas, 5 site, 6 app | site e app são "lojas" no sistema; confiável |
| `canal` | por onde a venda foi feita | `loja`, `PDV`, `site`, `APP`, `online` | 12 grafias para 3 canais; cada loja física tem a sua grafia; às vezes discorda do `loja_id` (`app` na loja 5; `online` nas lojas 5 e 6). Pouco confiável |
| `vendedor` | quem vendeu (texto livre) | `Bia`, `Larissa R.`, `THIAGO` | vazio no online (normal); 48 nomes, com a mesma pessoa escrita de vários jeitos |
| `cliente_ref` | identificação do cliente, como foi digitada | e-mail, CPF, celular, `NAO INFORMADO` | campo único para CPF, e-mail ou telefone; 3.884 vendas sem cliente (só lojas físicas) |
| `cliente_tipo` / `cliente_valor` | classificação e valor limpo do `cliente_ref` (feitos pelo `padronizar.py`) | `email`, `cpf`, `numero`, `telefone` | `numero` (5.747) = só dígitos, pode ser CPF ou celular; o app identifica quase sempre por `numero`, o site por e-mail |
| `total` | valor total do pedido | `589.70` | **app grava em centavos**: na loja 6 a mediana é 22.990 (R$ 229,90); os itens do app também estão em centavos, então o total "bate" com os itens e isso não serve para detectar; 201 negativos (devoluções) |
| `desconto` | desconto em R$ | `0.00`, vazio | 4.898 vazios |
| `frete` | frete em R$ | `14.90`, `29.90` | só na loja 5 (site) |
| `status` | situação | `PAGO`, `pago`, `aprovado`, vazio, `aguardando`, `cancelado`, `CANCELADA`, `estornado`, `chargeback` | 10 grafias; 949 vazios |
| `criado_em` | data e hora da venda | `2025-01-01 15:47:42+00:00` | em UTC (ver abaixo); de 01/01/2025 a 01/10/2026 |
| `obs` | observação livre | `troca` (328), `devolucao` (201), `sfs pinheiros` (56), `teste` (18), `lancamento retroativo` (16) | guarda troca, devolução, ship from store, testes e relançamentos da Batel |
| `pedido_externo` | número do pedido no site (`S10...`) ou no app (`APP-...`) | `S100234`, `APP-7001324` | só no online; 76 repetidos (abaixo) |
| `compra_id` | liga a venda à tabela `compras` (pontos) | `908.0` | só preenchido no online (lojas 5 e 6) |

**Colunas de `itens`**

| Coluna | O que é | Problemas vistos |
|---|---|---|
| `sku` | código da peça | 4 formatos: longo `VLT-...` (24.583), curto `CAM004BM` (5.631), com espaço (630), `DIVERSO` sem código (266) |
| `qtd` | quantidade | `-1` em 201 itens (devoluções); 96, 139, 109 e 56 são de um pedido corporativo de 400 camisetas (R$ 23.960, loja 1) |
| `preco_unit` | preço unitário | também em centavos no app |

**Datas**
- **`criado_em` está em UTC** (3 horas à frente de Brasília): `padronizar.py` converte todas as datas para UTC. Ex.: `2025-01-01 12:47:42` (Brasília, no original) vira `2025-01-01 15:47:42+00:00`. Para agrupar por dia ou mês (faturamento), converter antes para `America/Sao_Paulo`: agrupando em UTC, 73 vendas feitas à noite no último dia do mês caem no mês seguinte.

**Pedidos do app registrados duas vezes**
- **Fato:** 76 pedidos do app (`pedido_externo` `APP-...`) aparecem 2 vezes: 44 como aguardando → pago e 32 como pago → pago. Nos 76, as duas linhas têm o mesmo valor, a mesma loja e os mesmos itens, com menos de 90 segundos de diferença. É uma compra só registrada duas vezes (a causa não aparece nos dados; perguntar ao Diego se quiser confirmar).
- **Decidido (V5):** manter só a linha mais recente de cada pedido (está sempre paga). Medir o impacto no faturamento.
- **→ Matias (pontos):** cada cópia gerou um `compra_id` próprio, então o cliente provavelmente ganhou pontos do Vulto+ em dobro nesses 76 pedidos. Afeta o passivo de pontos.

**Frete do app escondido no total**
- **Fato:** em 803 vendas o `total` não é igual a itens − desconto + frete; 723 são do app (loja 6) e 74 são pedidos com canal `app`/`online` registrados na loja 5. A diferença é quase sempre um valor de frete: 9,90 / 14,90 / 19,90 (185 vendas, em reais) ou 990 / 1.490 / 1.990 (612 vendas, o mesmo frete em centavos). O app cobra frete, mas não grava na coluna `frete`: ele fica dentro do `total`. Restam ~6 casos isolados (5 da Batel, 1 loja física).
- **Para a decisão do frete:** se o frete sair do faturamento, no app ele precisa ser calculado (total − itens + desconto), porque a coluna `frete` está vazia.

**O app nem sempre gravou em centavos**
- **Fato:** o app mudou de formato em ago/2025. De jan a ago/2025 grava em **reais** (~700 vendas); a partir de ago/2025, em **centavos** (~2.300). Em jun/2026 (mês do bug dos pontos, app v3.4), 15 vendas voltam a sair em reais. Há também ~200 vendas com canal `app`/`APP`/`online` registradas na loja 5, em centavos.
- Comparando o preço de cada item com o do catálogo, os valores caem em duas faixas bem separadas (~1× ou ~100×); só 24 vendas ficam numa faixa ambígua (total entre R$ 1.500 e R$ 5.000).
- **Decidido (V1):** detectar os centavos venda por venda (pelo preço dos itens × catálogo e pela faixa do total), e não por canal ou loja: "todo app ÷ 100" erraria ~700 vendas.

**Status vazio**
- **Fato:** as 949 vendas com status vazio são todas de lojas físicas e de jan–fev/2025. Nesse período não existe nenhuma venda "pago" em loja física (a primeira é de 01/03/2025), mas as canceladas já eram marcadas (21). Entre as vazias há 11 trocas e devoluções.
- **Decidido (V2, como premissa):** o PDV só passou a gravar "pago" em mar/2025; status vazio = venda normal (paga). Não está escrito em lugar nenhum: entrou no registro como premissa.

**Outros avisos e hipóteses**
- **→ Matias (pontos):** `compra_id` só existe nas vendas online. As vendas das lojas físicas com CPF não apontam para `compras`: os pontos delas, se existirem, foram lançados por outro caminho.
- **→ Matias (clientes):** dos `cliente_tipo = numero` (só dígitos, CPF ou celular?), 4.155 batem com um CPF da planilha e só 75 com um telefone. Na prática, a maioria é CPF.
- **Hipótese:** para o canal, confiar no `loja_id` e não no `canal` (proposta do README do Matias; o cruzamento canal × loja é a evidência).

**Problemas das vendas (lista única; as decisões estão no registro de decisões)**

| # | Problema | Quantas | Afeta |
|---|---|---|---|
| 1 | App grava valores em centavos desde ago/2025 (antes, em reais) | ~2.300 | faturamento (o principal) |
| 2 | Frete do app escondido no total | ~800 | faturamento (decisão do frete) |
| 3 | Pedidos do app registrados 2 vezes | 76 | faturamento e pontos |
| 4 | `status` com 10 grafias; 949 vazios (jan–fev/2025, lojas físicas) | todas | o que conta como venda |
| 5 | Devoluções como venda negativa | 201 | faturamento |
| 6 | Trocas registradas como venda nova | 328 | faturamento |
| 7 | Vendas de teste | 18 | faturamento |
| 8 | Relançamentos da Batel (março) | 16 | faturamento (risco de duplicar com a planilha manual) |
| 9 | Ship from store (pedido do site saindo de Pinheiros) | 56 | estoque |
| 10 | `canal` com 12 grafias e às vezes contra o `loja_id` | todas | faturamento por canal |
| 11 | Cliente em campo único (CPF, e-mail ou telefone); 3.884 sem cliente | todas | ligação com clientes e pontos |
| 12 | SKU em 4 formatos + `DIVERSO` | itens | ligação com catálogo e estoque |
| 13 | Datas em UTC | todas | faturamento por mês (73 vendas) |

**Ligações com as outras fontes**

| Ligação | Funciona? | Detalhe |
|---|---|---|
| `itens.venda_id` → `vendas` | 100% | todo item tem venda |
| `vendas.loja_id` → `lojas` | 100% | |
| `vendas.compra_id` → `compras` | 100% | nas 7.665 vendas que têm (só online) |
| `itens.sku` → estoque | 100% | o estoque usa os mesmos códigos das vendas |
| `itens.sku` → catálogo | 55% das linhas (878 de 1.913 códigos) | sistema e planilha de Compras escrevem os códigos de jeitos diferentes (ver `02_catalogo`) |
| `vendas.cliente` → planilha de clientes | ~70% dos identificados | CPF: 3.039 acham, 816 não; e-mail: 2.789 acham, 1.988 não; `numero`: 4.155 acham como CPF, 75 como telefone, 1.517 não |
| `atendimentos.pedido` → `vendas` | 85% (574 de 678) | 245 pelo número do site/app, 329 por um número solto que bate com o id interno (pode ser coincidência: ligação menos confiável) |